# Maven Roasters Coffee Shop — Mini Project

You've explored two retail datasets and practised the core principles of good data visualisation. Now it's time to go solo.

This project uses a real-world-style dataset from **Maven Roasters**, a fictitious NYC coffee shop with three locations. Your job is to explore the data and answer 7 business questions using charts.

---

### How this notebook works

Each task follows the same structure:

1. **Business question**, the columns you'll need
2. **Your basic chart** — a blank cell for your first attempt. You need to decide which chart to use.
3. **Things to look out for + optional extras** — hints to improve your chart
4. **Your improved chart** — a blank cell to apply the improvements


Try each step yourself before peeking at the answers!

---

### Dataset
**Coffee Shop Sales — Maven Roasters**  
Downloaded from Kaggle: https://www.kaggle.com/datasets/ahmedabbas757/coffee-sales

~149,000 transactions from Jan–Jun 2023 across three NYC locations.

| Column | Description |
|--------|-------------|
| `transaction_id` | Unique ID per transaction |
| `transaction_date` | Date of purchase (DD/MM/YYYY) |
| `transaction_time` | Time of purchase (HH:MM:SS) |
| `transaction_qty` | Number of items purchased |
| `store_location` | Store name: Astoria, Hell's Kitchen, or Lower Manhattan |
| `unit_price` | Price per item ($) |
| `product_category` | High-level category, e.g. Coffee, Tea, Bakery |
| `product_type` | More specific type, e.g. Brewed Chai Tea, Scone |
| `product_detail` | Full product name |

> There is no `revenue` column — the Setup cell creates it for you: `revenue = transaction_qty × unit_price`

---
## Setup — Run This First

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv('coffee_shop_sales.csv')

# Note: dates in this file are DD/MM/YYYY format, so we use dayfirst=True
df['transaction_date'] = pd.to_datetime(df['transaction_date'], dayfirst=True)
df['transaction_time'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.time
df['month']      = df['transaction_date'].dt.month
df['month_name'] = df['transaction_date'].dt.strftime('%b')
df['day_name']   = df['transaction_date'].dt.strftime('%a')
df['hour']       = pd.to_datetime(df['transaction_time'].astype(str)).dt.hour

# Create revenue column
df['revenue'] = df['transaction_qty'] * df['unit_price']

print(f"✅ Loaded {len(df):,} transactions from {df['transaction_date'].min().date()} to {df['transaction_date'].max().date()}")
print(f"   Stores:     {list(df['store_location'].unique())}")
print(f"   Categories: {list(df['product_category'].unique())}")
df.head()

---
## Task 1 — Which product category brings in the most revenue?

Maven Roasters sells coffee, tea, bakery items, drinking chocolate, and more. The business wants a quick ranking of which categories are driving revenue.

**Columns you'll need:** `product_category`, `revenue`

In [ ]:
cat_revenue = df.groupby('product_category')['revenue'].sum()

# Your basic chart here


### Things to look out for
- Is there one clear leader, or are the categories close in revenue?
- Are the category name labels readable at the current rotation?
- A **horizontal** bar chart often works better with text labels — try `kind='barh'`
- Nothing in the basic chart draws the eye to the top performer

### Optional extras to try
- Sort bars by value and switch to horizontal: `sort_values(ascending=True)` + `plt.barh()`
- Highlight the top category in amber (`#E69F00`), use teal (`#009E73`) for the rest
- Add revenue labels on each bar: `plt.text(v * 1.01, i, f'${v:,.0f}', va='center')`
- Write an insight-driven title — what's the actual finding?
- Use `sns.despine()` to remove the chart border

In [ ]:
cat_revenue = df.groupby('product_category')['revenue'].sum().sort_values(ascending=True)

# Your improved chart here


---
## Task 2 — How has revenue trended over the six months?

The business wants to see whether revenue is growing and spot any month-to-month patterns. Which chart would suit here if we want to emphasise continuity and direction over time?

**Columns you'll need:** `month_name`, `revenue`

In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
monthly = df.groupby('month_name')['revenue'].sum().reindex(month_order) #`month_order` list and `.reindex()` - without this, months sort alphabetically

# Your basic chart here


### Things to look out for
- Is the trend consistently upward, or are there any dips?
- The peak month isn't annotated — a reader has to squint at the y-axis to get the value
- The y-axis shows raw numbers — `$K` formatting would be much cleaner
- A plain line looks flat and impersonal — colour and line weight make it feel more polished

### Optional extras to try
- Colour the line teal (`#009E73`) with `linewidth=2.5`
- Mark the peak month with a dot: `plt.scatter(peak_month, peak_val, color='#D55E00', s=100, zorder=5)`
- Label the peak with `plt.text()`
- Calculate the % growth from Jan to Jun and add it as a subtitle with `plt.suptitle()`
- Use `sns.set_style('whitegrid')` for a cleaner background
- Format the y-axis as `$K`: `plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K')`

In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
monthly = df.groupby('month_name')['revenue'].sum().reindex(month_order)

# Your improved chart here


---
## Task 3 — What does the spread of transaction values look like?


The finance team wants to understand the shape of individual transaction values. Are most transactions small with a few large outliers? Or is spending fairly evenly spread? This tells them a lot about typical customer behaviour.

**Columns you'll need:** `revenue` (one row per transaction — no groupby needed)

In [ ]:
# Your basic chart here


### Things to look out for
- Is the distribution **skewed**? (Most values clustered on the left with a long tail to the right is very common for transaction data)
- Are there any outliers pulling the x-axis far to the right and squashing everything else?
- Where is the bulk of transactions concentrated — under $5, $5–10, $10+?

### Optional extras to try
- Cap the x-axis to focus on the bulk of transactions and ignore extreme outliers:  
  `plt.xlim(0, 20)` — try different values to find where most transactions sit
- Add a vertical line for the **median**: `plt.axvline(df['revenue'].median(), color='#D55E00', linestyle='--', label='Median')`
- Add a vertical line for the **mean** in a different colour — if the mean is much higher than the median, the distribution is skewed
- Change the bar colour to teal and add `edgecolor='white'` for cleaner bars
- Increase `bins=` to get a finer view of the shape

In [ ]:
# Your improved chart here


---
## Task 4 — Is there a relationship between unit price and quantity ordered?

The buying team wants to know: do customers buy more units when items are cheaper, or does quantity stay the same regardless of price? Understanding this relationship helps with pricing and promotions.


> **Heads up — overplotting:** With 149,000 rows, plotting all points at once creates a solid blob. Use a random **sample** of a few thousand rows instead.

**Columns you'll need:** `unit_price`, `transaction_qty`

In [ ]:
sample = df.sample(3000, random_state=42)

# Your basic chart here


### Things to look out for
- Can you see any pattern, or do the points form a grid-like structure? (This happens when prices and quantities only take a small number of distinct values)
- Are there any obvious outliers — unusually large quantities or very high prices?
- Lots of overlapping points make it hard to see density — try reducing `alpha` further or point size `s`

### Optional extras to try
- Reduce point size: `s=15` and alpha: `alpha=0.2` to reduce the overlap blob
- Add a **jitter** to spread overlapping points: `unit_price + np.random.uniform(-0.1, 0.1, len(sample))`
- Add a trend line using `np.polyfit`:  
  ```python
  m, b = np.polyfit(sample['unit_price'], sample['transaction_qty'], 1)
  x_line = np.linspace(sample['unit_price'].min(), sample['unit_price'].max(), 100)
  plt.plot(x_line, m * x_line + b, color='#D55E00', linewidth=2, linestyle='--')
  ```
- Colour points by `product_category` to see if different categories cluster

In [ ]:
sample = df.sample(5000, random_state=42)

# Your improved chart here


---
## Task 5 — Which store and category combinations generate the most revenue?

The operations team wants to see revenue broken down by both **store location** and **product category** at the same time. 

**Columns you'll need:** `store_location`, `product_category`, `revenue`

In [ ]:
# Your basic chart here


### Things to look out for
- The raw numbers in the basic chart are hard to read — `$45678.90` in a small cell is cluttered
- The default colour palette (`viridis`) works, but doesn't naturally suggest "high is good"
- Are some category × store combinations noticeably stronger or weaker than the rest?
- Do all three stores show a similar pattern, or does one store have a different product mix?

### Optional extras to try
- Format annotations as `$K` for cleaner cells:  
  `annot_labels = pivot.map(lambda v: f'${v/1000:.0f}K')`  
  then pass `annot=annot_labels, fmt=''` to `sns.heatmap()`
- Use `cmap='YlOrRd'` (yellow → orange → red) for a more intuitive "low to high" colour scale
- Add `linewidths=0.5, linecolor='white'` to separate cells clearly
- Set `annot_kws={'size': 10, 'weight': 'bold'}` for bolder labels
- Write a title that states what the pattern is, not just what the chart shows

In [ ]:
# Your improved chart here


---
## Task 6 — How has each category's share of revenue shifted over time?

The strategy team wants to know whether the **mix** of product categories is changing — is coffee's dominance growing or shrinking? Are any categories gaining share month by month?


**Columns you'll need:** `month_name`, `product_category`, `revenue`

In [ ]:
# Your basic chart here


### Things to look out for
- Coffee dominates the chart so much it's hard to see what the other categories are doing
- The default palette mixes colours unpredictably — consider assigning them deliberately
- Is the *mix* changing, or are all categories just growing proportionally together?
- A **100% stacked area** (`stacked=True` + normalised values) would answer the share question better

### Optional extras to try
- Normalise to 100% so each month's total adds up to 100%:  
  `pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100`
- Assign colours deliberately using a palette dictionary
- Format y-axis as `%` if using the normalised version
- Add a legend outside the chart so it doesn't overlap the areas:  
  `plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')`
- Use `alpha=0.85` on the areas so overlapping regions are still visible

In [ ]:
# Your improved chart here


---
## Task 7 — Your own question

The final task is entirely yours. Explore the dataset, find something interesting, and visualise it — choosing the chart type that fits the question best.

Some ideas if you need a starting point:

| Idea | Suggested chart | Columns to explore |
|------|-----------------|--------------------|
| Which hours of the day are the busiest? | Bar chart | `hour`, `transaction_id` |
| How do the three stores compare month by month? | Line chart (small multiples) | `store_location`, `month_name`, `revenue` |
| Which days of the week drive the most transactions? | Bar chart | `day_name`, `transaction_id` |
| Does the product mix vary by store? | Heatmap or stacked bar | `store_location`, `product_category` |
| Which specific products generate the most revenue? | Horizontal bar (top 10) | `product_detail`, `revenue` |

In [ ]:
# Your basic chart here


### Before improving your chart, ask yourself:
- Does your title state the **finding**, not just describe the chart?
- Is the **chart type** the right one for your question? 
- Is colour doing a **job** — or is it just decoration?
- Would a reader understand the chart **without reading your code**?
- Are axis labels and units clear?

### Optional extras to try
- Add value labels or annotations to key data points
- Use `sns.despine()` and `plt.tight_layout()` for a clean finish
- If comparing multiple groups, consider whether small multiples would be clearer than a single busy chart

In [ ]:
# Your improved chart here


---
## Reflection

Before you finish, take a moment to answer these:

1. **Which chart type is the hardest for you to understand/apply?** 
2. **Which task was hardest?** Was it the code, the data wrangling, or deciding what to show?
3. **Did anything surprise you** in the data?
4. **What one business recommendation** would you make to Maven Roasters based on your findings?

---

## Quick Reference

**Chart type guide:**

| Question type | Chart type |
|---------------|------------|
| Compare totals across categories | Horizontal bar chart |
| Show a trend over time | Line chart |
| Understand the shape of a variable | Histogram |
| Explore a relationship between two numbers | Scatter plot |
| Compare values across two dimensions | Heatmap |
| Show composition changing over time | Stacked area chart |

**Colours (Okabe-Ito palette):**

| Use | Hex |
|-----|-----|
| Highlight / top value | `#E69F00` |
| Base / positive | `#009E73` |
| Warning / annotation | `#D55E00` |
| Series 1 — Blue | `#0072B2` |
| Series 2 — Coral | `#e76f51` |
| Series 3 — Yellow | `#e9c46a` |

**Useful one-liners:**
```python
sns.despine()                             # Remove top and right border
plt.tight_layout()                        # Fix overlapping labels
plt.xticks(rotation=45, ha='right')       # Rotate x-axis labels
plt.gca().yaxis.set_major_formatter(      # Format y-axis as $K
    plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.axhline(val, linestyle='--', color='grey')          # Horizontal reference line
plt.axvline(val, linestyle='--', color='#D55E00')       # Vertical reference line
plt.axvspan(x1, x2, alpha=0.1, color='orange')         # Shaded region
plt.text(x, y, 'label', ha='center', fontsize=9)       # Text annotation
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left') # Legend outside chart
```